In [5]:
import numpy as np
import matplotlib.pyplot as plt
import pandas as pd
import seaborn as sns

import warnings
warnings.filterwarnings("ignore")

pd.set_option("display.max_columns", None)

In [2]:
df = pd.read_csv('dataset/train.csv')

In [3]:
df.drop(columns=["id"], inplace=True)

In [6]:
df.head()

,health_condition,sleep_duration,heart_rate,bmi,calorie_expenditure,step_count,exercise_duration,water_intake,diet_type,stress_level,sleep_quality,physical_activity_level,smoking_alcohol,gender
0,unhealthy,5.22,70.6,25.66,2174.0,1326.0,19.8,1.86,veg,high,average,sedentary,yes,female
1,at-risk,5.53,71.3,25.84,1966.0,9891.0,49.9,1.26,non-veg,low,average,moderate,yes,other
2,unhealthy,5.29,75.4,24.54,2688.0,14216.0,38.1,1.60,veg,high,poor,active,yes,male
3,unhealthy,4.70,77.2,23.13,2630.0,7174.0,59.9,2.02,veg,high,average,active,occasional,female
4,at-risk,7.23,73.4,28.44,2560.0,6584.0,46.0,2.25,veg,NaN,average,sedentary,NaN,male


In [7]:
def create_features(data):

    data = data.copy()

    # BMI category
    data["bmi_category"] = pd.cut(
        data["bmi"],
        bins=[0, 18.5, 25, 30, np.inf],
        labels=[
            "underweight",
            "normal",
            "overweight",
            "obese"
        ]
    )

    # Sleep category
    data["sleep_category"] = pd.cut(
        data["sleep_duration"],
        bins=[0, 6, 8, np.inf],
        labels=[
            "insufficient",
            "recommended",
            "long"
        ]
    )

    # Daily step category
    data["step_category"] = pd.cut(
        data["step_count"],
        bins=[0, 5000, 10000, np.inf],
        labels=[
            "low",
            "moderate",
            "high"
        ]
    )

    # Exercise category
    data["exercise_category"] = pd.cut(
        data["exercise_duration"],
        bins=[-1, 20, 45, np.inf],
        labels=[
            "low",
            "moderate",
            "high"
        ]
    )

    # Hydration category
    data["hydration_category"] = pd.cut(
        data["water_intake"],
        bins=[0, 1.5, 2.5, np.inf],
        labels=[
            "low",
            "adequate",
            "high"
        ]
    )

    # Combined activity score
    data["activity_score"] = (
        data["step_count"] *
        data["exercise_duration"]
    )

    # Interaction between sleep and exercise
    data["sleep_activity_interaction"] = (
        data["sleep_duration"] *
        data["exercise_duration"]
    )

    return data

In [8]:
df_featured = create_features(df)

In [9]:
df_featured.head()

,health_condition,sleep_duration,heart_rate,bmi,calorie_expenditure,step_count,exercise_duration,water_intake,diet_type,stress_level,sleep_quality,physical_activity_level,smoking_alcohol,gender,bmi_category,sleep_category,step_category,exercise_category,hydration_category,activity_score,sleep_activity_interaction
0,unhealthy,5.22,70.6,25.66,2174.0,1326.0,19.8,1.86,veg,high,average,sedentary,yes,female,overweight,insufficient,low,low,adequate,26254.8,103.356
1,at-risk,5.53,71.3,25.84,1966.0,9891.0,49.9,1.26,non-veg,low,average,moderate,yes,other,overweight,insufficient,moderate,high,low,493560.9,275.947
2,unhealthy,5.29,75.4,24.54,2688.0,14216.0,38.1,1.60,veg,high,poor,active,yes,male,normal,insufficient,high,moderate,adequate,541629.6,201.549
3,unhealthy,4.70,77.2,23.13,2630.0,7174.0,59.9,2.02,veg,high,average,active,occasional,female,normal,insufficient,moderate,high,adequate,429722.6,281.530
4,at-risk,7.23,73.4,28.44,2560.0,6584.0,46.0,2.25,veg,NaN,average,sedentary,NaN,male,overweight,recommended,moderate,high,adequate,302864.0,332.580


In [10]:
X = df_featured.drop(
    columns="health_condition"
)

y = df_featured[
    "health_condition"
]

In [11]:
from sklearn.model_selection import train_test_split
X_train, X_test, y_train, y_test = train_test_split(
    X,
    y,
    test_size=0.20,
    random_state=42,
    stratify=y
)

In [12]:
numerical_features = (
    X_train
    .select_dtypes(
        include=["number"]
    )
    .columns
    .tolist()
)

categorical_features = (
    X_train
    .select_dtypes(
        exclude=["number"]
    )
    .columns
    .tolist()
)

In [13]:
print(
    "Numerical features:"
)

print(
    numerical_features
)

print(
    "\nCategorical features:"
)

print(
    categorical_features
)

Numerical features:
['sleep_duration', 'heart_rate', 'bmi', 'calorie_expenditure', 'step_count', 'exercise_duration', 'water_intake', 'activity_score', 'sleep_activity_interaction']

Categorical features:
['diet_type', 'stress_level', 'sleep_quality', 'physical_activity_level', 'smoking_alcohol', 'gender', 'bmi_category', 'sleep_category', 'step_category', 'exercise_category', 'hydration_category']


In [14]:
from sklearn.pipeline import Pipeline

from sklearn.compose import (
    ColumnTransformer
)

from sklearn.impute import (
    SimpleImputer
)

from sklearn.preprocessing import (
    OneHotEncoder,
    StandardScaler
)

In [15]:
# numerical preprocessing pipeline
numerical_pipeline = Pipeline(
    steps=[
        (
            "imputer",
            SimpleImputer(
                strategy="median" # fill missing values with median for numerical features
            )
        ),

        (
            "scaler",
            StandardScaler() # scale numerical features
        )
    ]
)

# categorical preprocessing pipeline
categorical_pipeline = Pipeline(
    steps=[
        (
            "imputer",
            SimpleImputer(
                strategy="most_frequent" # fill missing values with the most frequent value for categorical features
            )
        ),

        (
            "encoder", # encode categorical features using one-hot encoding
            OneHotEncoder(
                handle_unknown="ignore" # ignore unknown categories during encoding
            )
        )
    ]
)

# Combine numerical and categorical pipelines into a single preprocessor
preprocessor = ColumnTransformer(
    transformers=[
        (
            "numerical",
            numerical_pipeline,
            numerical_features
        ),

        (
            "categorical",
            categorical_pipeline,
            categorical_features
        )
    ]
)

In [16]:
from sklearn.ensemble import RandomForestClassifier

In [17]:
random_forest_model = Pipeline(
    steps=[
        (
            "preprocessor",
            preprocessor
        ),

        (
            "classifier",
            RandomForestClassifier(
                n_estimators=200,
                max_depth=20,
                min_samples_split=10,
                min_samples_leaf=5,
                class_weight="balanced",
                n_jobs=-1,
                random_state=42
            )
        )
    ]
)

In [19]:
random_forest_model.fit(
    X_train,
    y_train
)

,"steps steps: list of tuplesList of (name of step, estimator) tuples that are to be chained insequential order. To be compatible with the scikit-learn API, all stepsmust define `fit`. All non-last steps must also define `transform`. See:ref:`Combining Estimators <combining_estimators>` for more details.","[('preprocessor', ...), ('classifier', ...)]"
,"transform_input transform_input: list of str, default=NoneThe names of the :term:`metadata` parameters that should be transformed by thepipeline before passing it to the step consuming it.This enables transforming some input arguments to ``fit`` (other than ``X``)to be transformed by the steps of the pipeline up to the step which requiresthem. Requirement is defined via :ref:`metadata routing <metadata_routing>`.For instance, this can be used to pass a validation set through the pipeline.You can only set this if metadata routing is enabled, which youcan enable using ``sklearn.set_config(enable_metadata_routing=True)``... versionadded:: 1.6",None
,"memory memory: str or object with the joblib.Memory interface, default=NoneUsed to cache the fitted transformers of the pipeline. The last stepwill never be cached, even if it is a transformer. By default, nocaching is performed. If a string is given, it is the path to thecaching directory. Enabling caching triggers a clone of the transformersbefore fitting. Therefore, the transformer instance given to thepipeline cannot be inspected directly. Use the attribute ``named_steps``or ``steps`` to inspect estimators within the pipeline. Caching thetransformers is advantageous when fitting is time consuming. See:ref:`sphx_glr_auto_examples_neighbors_plot_caching_nearest_neighbors.py`for an example on how to enable caching.",None
,"verbose verbose: bool, default=FalseIf True, the time elapsed while fitting each step will be printed as itis completed.",False
Name,Type,Value
"classes_ classes_: ndarray of shape (n_classes,)The classes labels. Only exist if the last step of the pipeline is aclassifier.","ndarray[object](3,)","['at-risk','fit','unhealthy']"
"feature_names_in_ feature_names_in_: ndarray of shape (`n_features_in_`,)Names of features seen during :term:`fit`. Only defined if theunderlying estimator exposes such an attribute when fit... versionadded:: 1.0","ndarray[object](20,)","['sleep_duration','heart_rate','bmi',...,'hydration_category', 'activity_score','sleep_activity_interaction']"
n_features_in_ n_features_in_: intNumber of features seen during :term:`fit`. Only defined if theunderlying first estimator in `steps` exposes such an attributewhen fit... versionadded:: 0.24,int,20
,"transformers transformers: list of tuplesList of (name, transformer, columns) tuples specifying thetransformer objects to be applied to subsets of the data.name : str Like in Pipeline and FeatureUnion, this allows the transformer and its parameters to be set using ``set_params`` and searched in grid search.transformer : {'drop', 'passthrough'} or estimator Estimator must support :term:`fit` and :term:`transform`. Special-cased strings 'drop' and 'passthrough' are accepted as well, to indicate to drop the columns or to pass them through untransformed, respectively.columns : str, array-like of str, int, array-like of int, array-like of bool, slice or callable Indexes the data on its second axis. Integers are interpreted as positional columns, while strings can reference DataFrame columns by name. A scalar string or int should be used where ``transformer`` expects X to be a 1d array-like (vector), otherwise a 2d array will be passed to the transformer. A callable is passed the input data `X` and can return any of the above. To select multiple columns by name or dtype, you can use :obj:`make_column_selector`.","[('numerical', ...), ('categorical', ...)]"
,"remainder remainder: {'drop', 'passthrough'} or estimator, default='drop'By default, only the specified columns in `transformers` aretransformed and combined in the output, and the non-specifiedcolumns ar

In [20]:
y_pred_rf = random_forest_model.predict(
    X_test
)

In [21]:
from sklearn.metrics import (
    accuracy_score,
    balanced_accuracy_score,
    classification_report,
    confusion_matrix,
    f1_score
)

In [22]:
print(
    "Accuracy:",
    round(
        accuracy_score(
            y_test,
            y_pred_rf
        ),
        4
    )
)

print(
    "Balanced Accuracy:",
    round(
        balanced_accuracy_score(
            y_test,
            y_pred_rf
        ),
        4
    )
)

print(
    "Macro F1:",
    round(
        f1_score(
            y_test,
            y_pred_rf,
            average="macro"
        ),
        4
    )
)

print(
    "\nClassification Report:\n"
)

print(
    classification_report(
        y_test,
        y_pred_rf
    )
)

Accuracy: 0.9177
Balanced Accuracy: 0.9001
Macro F1: 0.8176

Classification Report:

              precision    recall  f1-score   support

     at-risk       0.98      0.92      0.95    118512
         fit       0.64      0.88      0.74      7961
   unhealthy       0.66      0.90      0.76     11545

    accuracy                           0.92    138018
   macro avg       0.76      0.90      0.82    138018
weighted avg       0.93      0.92      0.92    138018



In [24]:
print(
    classification_report(
        y_test,
        y_pred_rf
    )
)

              precision    recall  f1-score   support

     at-risk       0.98      0.92      0.95    118512
         fit       0.64      0.88      0.74      7961
   unhealthy       0.66      0.90      0.76     11545

    accuracy                           0.92    138018
   macro avg       0.76      0.90      0.82    138018
weighted avg       0.93      0.92      0.92    138018



In [25]:
def predict_student_health(
    model,
    student_data
):

    # Convert input dictionary into DataFrame
    student_df = pd.DataFrame(
        [student_data]
    )

    # Apply the same feature engineering
    student_df = create_features(
        student_df
    )

    # Predict health condition
    prediction = model.predict(
        student_df
    )[0]

    # Predict probabilities
    probabilities = model.predict_proba(
        student_df
    )[0]

    # Get class names
    classes = (
        model
        .named_steps["classifier"]
        .classes_
    )

    probability_results = {
        health_class:
        round(
            probability * 100,
            2
        )

        for health_class, probability
        in zip(
            classes,
            probabilities
        )
    }

    return (
        prediction,
        probability_results
    )

In [26]:
student = {

    "sleep_duration": 5.5,

    "heart_rate": 82,

    "bmi": 27.5,

    "calorie_expenditure": 1900,

    "step_count": 4500,

    "exercise_duration": 20,

    "water_intake": 1.8,

    "diet_type": "non-veg",

    "stress_level": "high",

    "sleep_quality": "poor",

    "physical_activity_level":
        "sedentary",

    "smoking_alcohol": "yes",

    "gender": "male"
}

In [27]:
prediction, probabilities = (
    predict_student_health(
        random_forest_model,
        student
    )
)

print(
    "Predicted Health Condition:",
    prediction
)

print(
    "\nPrediction Probabilities:"
)

for health_class, probability \
in probabilities.items():

    print(
        f"{health_class}: "
        f"{probability}%"
    )

Predicted Health Condition: unhealthy

Prediction Probabilities:
at-risk: 0.36%
fit: 0.0%
unhealthy: 99.64%


In [28]:
def get_prediction_confidence(
    probabilities
):

    confidence = max(
        probabilities.values()
    )

    if confidence >= 80:

        confidence_level = "High"

    elif confidence >= 60:

        confidence_level = "Moderate"

    else:

        confidence_level = "Low"

    return (
        confidence,
        confidence_level
    )

In [29]:
confidence, confidence_level = (
    get_prediction_confidence(
        probabilities
    )
)

print(
    "Prediction Confidence:",
    f"{confidence}%"
)

print(
    "Confidence Level:",
    confidence_level
)

Prediction Confidence: 99.64%
Confidence Level: High


In [33]:
from sklearn.preprocessing import (
    LabelEncoder
)

label_encoder = LabelEncoder()

y_train_encoded = (
    label_encoder.fit_transform(
        y_train
    )
)

y_test_encoded = (
    label_encoder.transform(
        y_test
    )
)

print(
    label_encoder.classes_
)

['at-risk' 'fit' 'unhealthy']


In [34]:
X_train_processed = preprocessor.fit_transform(
    X_train
)

X_test_processed = preprocessor.transform(
    X_test
)

print(
    X_train_processed.shape
)

(552070, 43)


In [35]:
from sklearn.model_selection import (
    RandomizedSearchCV
)

from xgboost import XGBClassifier

In [36]:
X_tune, _, y_tune, _ = train_test_split(
    X_train,
    y_train_encoded,
    train_size=200000,
    random_state=42,
    stratify=y_train_encoded
)

print(X_tune.shape)

(200000, 20)


In [38]:
from sklearn.base import clone

In [39]:
xgb_tuning_pipeline = Pipeline(
    steps=[
        (
            "preprocessor",
            clone(preprocessor)
        ),

        (
            "classifier",
            XGBClassifier(
                objective="multi:softprob",
                eval_metric="mlogloss",
                n_jobs=-1,
                random_state=42
            )
        )
    ]
)

In [40]:
xgb_parameters = {

    "classifier__n_estimators":
        [200, 300, 500, 700],

    "classifier__max_depth":
        [4, 6, 8, 10],

    "classifier__learning_rate":
        [0.03, 0.05, 0.1, 0.15],

    "classifier__subsample":
        [0.7, 0.8, 0.9, 1.0],

    "classifier__colsample_bytree":
        [0.7, 0.8, 0.9, 1.0],

    "classifier__min_child_weight":
        [1, 3, 5, 7],

    "classifier__gamma":
        [0, 0.1, 0.3],

    "classifier__reg_alpha":
        [0, 0.01, 0.1],

    "classifier__reg_lambda":
        [1, 2, 5]
}

In [41]:
xgb_search = RandomizedSearchCV(

    estimator=
        xgb_tuning_pipeline,

    param_distributions=
        xgb_parameters,

    n_iter=25,

    scoring="f1_macro",

    cv=3,

    verbose=2,

    random_state=42,

    n_jobs=1,

    return_train_score=True
)

In [42]:
xgb_search.fit(
    X_tune,
    y_tune
)

Fitting 3 folds for each of 25 candidates, totalling 75 fits
[CV] END classifier__colsample_bytree=0.7, classifier__gamma=0.1, classifier__learning_rate=0.1, classifier__max_depth=10, classifier__min_child_weight=3, classifier__n_estimators=500, classifier__reg_alpha=0.1, classifier__reg_lambda=1, classifier__subsample=1.0; total time=   8.9s
[CV] END classifier__colsample_bytree=0.7, classifier__gamma=0.1, classifier__learning_rate=0.1, classifier__max_depth=10, classifier__min_child_weight=3, classifier__n_estimators=500, classifier__reg_alpha=0.1, classifier__reg_lambda=1, classifier__subsample=1.0; total time=   7.9s
[CV] END classifier__colsample_bytree=0.7, classifier__gamma=0.1, classifier__learning_rate=0.1, classifier__max_depth=10, classifier__min_child_weight=3, classifier__n_estimators=500, classifier__reg_alpha=0.1, classifier__reg_lambda=1, classifier__subsample=1.0; total time=   7.7s
[CV] END classifier__colsample_bytree=0.7, classifier__gamma=0, classifier__learning_ra

,"estimator estimator: estimator objectAn object of that type is instantiated for each grid point.This is assumed to implement the scikit-learn estimator interface.Either estimator needs to provide a ``score`` function,or ``scoring`` must be passed.","Pipeline(step...=None, ...))])"
,"param_distributions param_distributions: dict or list of dictsDictionary with parameters names (`str`) as keys and distributionsor lists of parameters to try. Distributions must provide a ``rvs``method for sampling (such as those from scipy.stats.distributions).If a list is given, it is sampled uniformly.If a list of dicts is given, first a dict is sampled uniformly, andthen a parameter is sampled using that dict as above.","{'classifier__colsample_bytree': [0.7, 0.8, ...], 'classifier__gamma': [0, 0.1, ...], 'classifier__learning_rate': [0.03, 0.05, ...], 'classifier__max_depth': [4, 6, ...], ...}"
,"n_iter n_iter: int, default=10Number of parameter settings that are sampled. n_iter tradesoff runtime vs quality of the solution.",25
,"scoring scoring: str, callable, list, tuple or dict, default=NoneStrategy to evaluate the performance of the cross-validated model onthe test set.If `scoring` represents a single score, one can use:- a single string (see :ref:`scoring_string_names`);- a callable (see :ref:`scoring_callable`) that returns a single value;- `None`, the `estimator`'s :ref:`default evaluation criterion <scoring_api_overview>` is used.If `scoring` represents multiple scores, one can use:- a list or tuple of unique strings;- a callable returning a dictionary where the keys are the metric names and the values are the metric scores;- a dictionary with metric names as keys and callables as values.See :ref:`multimetric_grid_search` for an example.If None, the estimator's score method is used.",'f1_macro'
,"n_jobs n_jobs: int, default=NoneNumber of jobs to run in parallel.``None`` means 1 unless in a :obj:`joblib.parallel_backend` context.``-1`` means using all processors. See :term:`Glossary <n_jobs>`for more details... versionchanged:: v0.20 `n_jobs` default changed from 1 to None",1
,"cv cv: int, cross-validation generator or an iterable, default=NoneDetermines the cross-validation splitting strategy.Possible inputs for cv are:- None, to use the default 5-fold cross validation,- integer, to specify the number of folds in a `(Stratified)KFold`,- :term:`CV splitter`,- an iterable yielding (train, test) splits as arrays of indices.For integer/None inputs, if the estimator is a classifier and ``y`` iseither binary or multiclass, :class:`StratifiedKFold` is used. In allother cases, :class:`KFold` is used. These splitters are instantiatedwith `shuffle=False` so the splits will be the same across calls.Refer :ref:`User Guide <cross_validation>` for the variouscross-validation strategies that can be used here... versionchanged:: 0.22 ``cv`` default value if None changed from 3-fold to 5-fold.",3
,"verbose verbose: int, default = 0Controls the verbosity of information printed during fitting, with highervalues yielding more detailed logging.- 0 : no messages are printed;- >=1 : summary of the total number of fits;- >=2 : computation time for each fold and parameter candidate;- >=3 : fold indices and scores;- >=10 : parameter candidate indices and START messages before each fit.",2
,"random_state random_state: int, RandomState instance or None, default=NonePseudo random number generator state used for random uniform samplingfrom lists of possible values instead of scipy.stats distributions.Pass an int for reproducible output across multiplefunction calls.See :term:`Glossary <random_state>`.",42
,"return_train_score return_train_score: bool, default=FalseIf ``False``, the ``cv_results_`` attribute will not include trainingscores.Computing training scores is used to get insights on how differentparameter settings impact the overfitting/underfitting trade-off.However computing the scores on the training set can be computationallyexpensive and is not stric

In [43]:
print(
    "Best CV Macro F1:",
    round(
        xgb_search.best_score_,
        4
    )
)

print(
    "\nBest parameters:"
)

xgb_search.best_params_

Best CV Macro F1: 0.902

Best parameters:


{'classifier__subsample': 0.7,
 'classifier__reg_lambda': 1,
 'classifier__reg_alpha': 0,
 'classifier__n_estimators': 300,
 'classifier__min_child_weight': 7,
 'classifier__max_depth': 6,
 'classifier__learning_rate': 0.05,
 'classifier__gamma': 0.3,
 'classifier__colsample_bytree': 1.0}

In [44]:
best_xgb_model = (
    xgb_search.best_estimator_
)

best_xgb_model.fit(
    X_train,
    y_train_encoded
)

,"steps steps: list of tuplesList of (name of step, estimator) tuples that are to be chained insequential order. To be compatible with the scikit-learn API, all stepsmust define `fit`. All non-last steps must also define `transform`. See:ref:`Combining Estimators <combining_estimators>` for more details.","[('preprocessor', ...), ('classifier', ...)]"
,"transform_input transform_input: list of str, default=NoneThe names of the :term:`metadata` parameters that should be transformed by thepipeline before passing it to the step consuming it.This enables transforming some input arguments to ``fit`` (other than ``X``)to be transformed by the steps of the pipeline up to the step which requiresthem. Requirement is defined via :ref:`metadata routing <metadata_routing>`.For instance, this can be used to pass a validation set through the pipeline.You can only set this if metadata routing is enabled, which youcan enable using ``sklearn.set_config(enable_metadata_routing=True)``... versionadded:: 1.6",None
,"memory memory: str or object with the joblib.Memory interface, default=NoneUsed to cache the fitted transformers of the pipeline. The last stepwill never be cached, even if it is a transformer. By default, nocaching is performed. If a string is given, it is the path to thecaching directory. Enabling caching triggers a clone of the transformersbefore fitting. Therefore, the transformer instance given to thepipeline cannot be inspected directly. Use the attribute ``named_steps``or ``steps`` to inspect estimators within the pipeline. Caching thetransformers is advantageous when fitting is time consuming. See:ref:`sphx_glr_auto_examples_neighbors_plot_caching_nearest_neighbors.py`for an example on how to enable caching.",None
,"verbose verbose: bool, default=FalseIf True, the time elapsed while fitting each step will be printed as itis completed.",False
Name,Type,Value
"classes_ classes_: ndarray of shape (n_classes,)The classes labels. Only exist if the last step of the pipeline is aclassifier.","ndarray[int64](3,)","[0,1,2]"
"feature_names_in_ feature_names_in_: ndarray of shape (`n_features_in_`,)Names of features seen during :term:`fit`. Only defined if theunderlying estimator exposes such an attribute when fit... versionadded:: 1.0","ndarray[object](20,)","['sleep_duration','heart_rate','bmi',...,'hydration_category', 'activity_score','sleep_activity_interaction']"
n_features_in_ n_features_in_: intNumber of features seen during :term:`fit`. Only defined if theunderlying first estimator in `steps` exposes such an attributewhen fit... versionadded:: 0.24,int,20
,"transformers transformers: list of tuplesList of (name, transformer, columns) tuples specifying thetransformer objects to be applied to subsets of the data.name : str Like in Pipeline and FeatureUnion, this allows the transformer and its parameters to be set using ``set_params`` and searched in grid search.transformer : {'drop', 'passthrough'} or estimator Estimator must support :term:`fit` and :term:`transform`. Special-cased strings 'drop' and 'passthrough' are accepted as well, to indicate to drop the columns or to pass them through untransformed, respectively.columns : str, array-like of str, int, array-like of int, array-like of bool, slice or callable Indexes the data on its second axis. Integers are interpreted as positional columns, while strings can reference DataFrame columns by name. A scalar string or int should be used where ``transformer`` expects X to be a 1d array-like (vector), otherwise a 2d array will be passed to the transformer. A callable is passed the input data `X` and can return any of the above. To select multiple columns by name or dtype, you can use :obj:`make_column_selector`.","[('numerical', ...), ('categorical', ...)]"
,"remainder remainder: {'drop', 'passthrough'} or estimator, default='drop'By default, only the specified columns in `transformers` aretransformed and combined in the output, and the non-specifiedcolumns are dropped. (default of 

In [45]:
tuned_xgb_prediction = (
    best_xgb_model.predict(
        X_test
    )
)

print(
    "Accuracy:",
    round(
        accuracy_score(
            y_test_encoded,
            tuned_xgb_prediction
        ),
        4
    )
)

print(
    "Balanced Accuracy:",
    round(
        balanced_accuracy_score(
            y_test_encoded,
            tuned_xgb_prediction
        ),
        4
    )
)

print(
    "Macro F1:",
    round(
        f1_score(
            y_test_encoded,
            tuned_xgb_prediction,
            average="macro"
        ),
        4
    )
)

Accuracy: 0.9658
Balanced Accuracy: 0.858
Macro F1: 0.9043


In [46]:
rf_tuning_pipeline = Pipeline(
    steps=[
        (
            "preprocessor",
            clone(preprocessor)
        ),

        (
            "classifier",
            RandomForestClassifier(
                class_weight="balanced",
                n_jobs=-1,
                random_state=42
            )
        )
    ]
)

In [47]:
rf_parameters = {

    "classifier__n_estimators":
        [200, 300, 500],

    "classifier__max_depth":
        [15, 20, 25, None],

    "classifier__min_samples_split":
        [2, 5, 10, 20],

    "classifier__min_samples_leaf":
        [1, 2, 5, 10],

    "classifier__max_features":
        ["sqrt", "log2", 0.7],

    "classifier__class_weight":
        [
            "balanced",
            "balanced_subsample"
        ]
}

In [48]:
rf_search = RandomizedSearchCV(

    estimator=
        rf_tuning_pipeline,

    param_distributions=
        rf_parameters,

    n_iter=20,

    scoring="f1_macro",

    cv=3,

    verbose=2,

    random_state=42,

    n_jobs=1
)

In [49]:
rf_search.fit(
    X_tune,
    y_tune
)

Fitting 3 folds for each of 20 candidates, totalling 60 fits
[CV] END classifier__class_weight=balanced, classifier__max_depth=20, classifier__max_features=sqrt, classifier__min_samples_leaf=5, classifier__min_samples_split=10, classifier__n_estimators=200; total time=   8.0s
[CV] END classifier__class_weight=balanced, classifier__max_depth=20, classifier__max_features=sqrt, classifier__min_samples_leaf=5, classifier__min_samples_split=10, classifier__n_estimators=200; total time=   8.1s
[CV] END classifier__class_weight=balanced, classifier__max_depth=20, classifier__max_features=sqrt, classifier__min_samples_leaf=5, classifier__min_samples_split=10, classifier__n_estimators=200; total time=   7.2s
[CV] END classifier__class_weight=balanced_subsample, classifier__max_depth=20, classifier__max_features=sqrt, classifier__min_samples_leaf=5, classifier__min_samples_split=20, classifier__n_estimators=300; total time=  14.6s
[CV] END classifier__class_weight=balanced_subsample, classifier_

,"estimator estimator: estimator objectAn object of that type is instantiated for each grid point.This is assumed to implement the scikit-learn estimator interface.Either estimator needs to provide a ``score`` function,or ``scoring`` must be passed.",Pipeline(step...m_state=42))])
,"param_distributions param_distributions: dict or list of dictsDictionary with parameters names (`str`) as keys and distributionsor lists of parameters to try. Distributions must provide a ``rvs``method for sampling (such as those from scipy.stats.distributions).If a list is given, it is sampled uniformly.If a list of dicts is given, first a dict is sampled uniformly, andthen a parameter is sampled using that dict as above.","{'classifier__class_weight': ['balanced', 'balanced_subsample'], 'classifier__max_depth': [15, 20, ...], 'classifier__max_features': ['sqrt', 'log2', ...], 'classifier__min_samples_leaf': [1, 2, ...], ...}"
,"n_iter n_iter: int, default=10Number of parameter settings that are sampled. n_iter tradesoff runtime vs quality of the solution.",20
,"scoring scoring: str, callable, list, tuple or dict, default=NoneStrategy to evaluate the performance of the cross-validated model onthe test set.If `scoring` represents a single score, one can use:- a single string (see :ref:`scoring_string_names`);- a callable (see :ref:`scoring_callable`) that returns a single value;- `None`, the `estimator`'s :ref:`default evaluation criterion <scoring_api_overview>` is used.If `scoring` represents multiple scores, one can use:- a list or tuple of unique strings;- a callable returning a dictionary where the keys are the metric names and the values are the metric scores;- a dictionary with metric names as keys and callables as values.See :ref:`multimetric_grid_search` for an example.If None, the estimator's score method is used.",'f1_macro'
,"n_jobs n_jobs: int, default=NoneNumber of jobs to run in parallel.``None`` means 1 unless in a :obj:`joblib.parallel_backend` context.``-1`` means using all processors. See :term:`Glossary <n_jobs>`for more details... versionchanged:: v0.20 `n_jobs` default changed from 1 to None",1
,"cv cv: int, cross-validation generator or an iterable, default=NoneDetermines the cross-validation splitting strategy.Possible inputs for cv are:- None, to use the default 5-fold cross validation,- integer, to specify the number of folds in a `(Stratified)KFold`,- :term:`CV splitter`,- an iterable yielding (train, test) splits as arrays of indices.For integer/None inputs, if the estimator is a classifier and ``y`` iseither binary or multiclass, :class:`StratifiedKFold` is used. In allother cases, :class:`KFold` is used. These splitters are instantiatedwith `shuffle=False` so the splits will be the same across calls.Refer :ref:`User Guide <cross_validation>` for the variouscross-validation strategies that can be used here... versionchanged:: 0.22 ``cv`` default value if None changed from 3-fold to 5-fold.",3
,"verbose verbose: int, default = 0Controls the verbosity of information printed during fitting, with highervalues yielding more detailed logging.- 0 : no messages are printed;- >=1 : summary of the total number of fits;- >=2 : computation time for each fold and parameter candidate;- >=3 : fold indices and scores;- >=10 : parameter candidate indices and START messages before each fit.",2
,"random_state random_state: int, RandomState instance or None, default=NonePseudo random number generator state used for random uniform samplingfrom lists of possible values instead of scipy.stats distributions.Pass an int for reproducible output across multiplefunction calls.See :term:`Glossary <random_state>`.",42
,"refit refit: bool, str, or callable, default=TrueRefit an estimator using the best found parameters on the wholedataset.For multiple metric evaluation, this needs to be a `str` denoting thescorer that would be used to find the best parameters for refittingthe estimator at the end.Where there are considerations other than maximum scor

In [50]:
print(
    "Best RF CV Macro F1:",
    round(
        rf_search.best_score_,
        4
    )
)

print(
    rf_search.best_params_
)

Best RF CV Macro F1: 0.8988
{'classifier__n_estimators': 200, 'classifier__min_samples_split': 2, 'classifier__min_samples_leaf': 2, 'classifier__max_features': 'log2', 'classifier__max_depth': 25, 'classifier__class_weight': 'balanced_subsample'}


In [51]:
best_rf_model = (
    rf_search.best_estimator_
)

best_rf_model.fit(
    X_train,
    y_train_encoded
)

tuned_rf_prediction = (
    best_rf_model.predict(
        X_test
    )
)

In [53]:
rf_prediction_encoded = (
    label_encoder.transform(
        y_pred_rf
    )
)

In [65]:
tuned_comparison = pd.DataFrame({

    "Model": [
        "Original Random Forest",
        "Tuned Random Forest",
        #"Original XGBoost",
        "Tuned XGBoost"
    ],

    "Macro F1": [

        f1_score(
            y_test_encoded,
            rf_prediction_encoded,
            average="macro"
        ),

        f1_score(
            y_test_encoded,
            tuned_rf_prediction,
            average="macro"
        ),

        

        f1_score(
            y_test_encoded,
            tuned_xgb_prediction,
            average="macro"
        )
    ],
    "Accuracy": [

        accuracy_score(
            y_test_encoded,
            rf_prediction_encoded
        ),

        accuracy_score(
            y_test_encoded,
            tuned_rf_prediction
        ),

        accuracy_score(
            y_test_encoded,
            tuned_xgb_prediction
        )
    ],
    "Balanced Accuracy": [

        balanced_accuracy_score(
            y_test_encoded,
            rf_prediction_encoded
        ),

        balanced_accuracy_score(
            y_test_encoded,
            tuned_rf_prediction
        ),

        balanced_accuracy_score(
            y_test_encoded,
            tuned_xgb_prediction
        )
    ]
})

tuned_comparison.sort_values(
    "Macro F1",
    ascending=False
).round(4)

,Model,Macro F1,Accuracy,Balanced Accuracy
2,Tuned XGBoost,0.9043,0.9658,0.8580
1,Tuned Random Forest,0.8986,0.9624,0.8711
0,Original Random Forest,0.8176,0.9177,0.9001


In [61]:
random_forest_model_1 = Pipeline(
    steps=[
        (
            "preprocessor",
            preprocessor
        ),

        (
            "classifier",
            RandomForestClassifier()
        )
    ]
)

In [62]:
random_forest_model_1.fit(
    X_train,
    y_train
)

,"steps steps: list of tuplesList of (name of step, estimator) tuples that are to be chained insequential order. To be compatible with the scikit-learn API, all stepsmust define `fit`. All non-last steps must also define `transform`. See:ref:`Combining Estimators <combining_estimators>` for more details.","[('preprocessor', ...), ('classifier', ...)]"
,"transform_input transform_input: list of str, default=NoneThe names of the :term:`metadata` parameters that should be transformed by thepipeline before passing it to the step consuming it.This enables transforming some input arguments to ``fit`` (other than ``X``)to be transformed by the steps of the pipeline up to the step which requiresthem. Requirement is defined via :ref:`metadata routing <metadata_routing>`.For instance, this can be used to pass a validation set through the pipeline.You can only set this if metadata routing is enabled, which youcan enable using ``sklearn.set_config(enable_metadata_routing=True)``... versionadded:: 1.6",None
,"memory memory: str or object with the joblib.Memory interface, default=NoneUsed to cache the fitted transformers of the pipeline. The last stepwill never be cached, even if it is a transformer. By default, nocaching is performed. If a string is given, it is the path to thecaching directory. Enabling caching triggers a clone of the transformersbefore fitting. Therefore, the transformer instance given to thepipeline cannot be inspected directly. Use the attribute ``named_steps``or ``steps`` to inspect estimators within the pipeline. Caching thetransformers is advantageous when fitting is time consuming. See:ref:`sphx_glr_auto_examples_neighbors_plot_caching_nearest_neighbors.py`for an example on how to enable caching.",None
,"verbose verbose: bool, default=FalseIf True, the time elapsed while fitting each step will be printed as itis completed.",False
Name,Type,Value
"classes_ classes_: ndarray of shape (n_classes,)The classes labels. Only exist if the last step of the pipeline is aclassifier.","ndarray[object](3,)","['at-risk','fit','unhealthy']"
"feature_names_in_ feature_names_in_: ndarray of shape (`n_features_in_`,)Names of features seen during :term:`fit`. Only defined if theunderlying estimator exposes such an attribute when fit... versionadded:: 1.0","ndarray[object](20,)","['sleep_duration','heart_rate','bmi',...,'hydration_category', 'activity_score','sleep_activity_interaction']"
n_features_in_ n_features_in_: intNumber of features seen during :term:`fit`. Only defined if theunderlying first estimator in `steps` exposes such an attributewhen fit... versionadded:: 0.24,int,20
,"transformers transformers: list of tuplesList of (name, transformer, columns) tuples specifying thetransformer objects to be applied to subsets of the data.name : str Like in Pipeline and FeatureUnion, this allows the transformer and its parameters to be set using ``set_params`` and searched in grid search.transformer : {'drop', 'passthrough'} or estimator Estimator must support :term:`fit` and :term:`transform`. Special-cased strings 'drop' and 'passthrough' are accepted as well, to indicate to drop the columns or to pass them through untransformed, respectively.columns : str, array-like of str, int, array-like of int, array-like of bool, slice or callable Indexes the data on its second axis. Integers are interpreted as positional columns, while strings can reference DataFrame columns by name. A scalar string or int should be used where ``transformer`` expects X to be a 1d array-like (vector), otherwise a 2d array will be passed to the transformer. A callable is passed the input data `X` and can return any of the above. To select multiple columns by name or dtype, you can use :obj:`make_column_selector`.","[('numerical', ...), ('categorical', ...)]"
,"remainder remainder: {'drop', 'passthrough'} or estimator, default='drop'By default, only the specified columns in `transformers` aretransformed and combined in the output, and the non-specifiedcolumns ar

In [63]:
y_pred_rf = random_forest_model_1.predict(
    X_test
)

In [64]:
print(
    "Accuracy:",
    round(
        accuracy_score(
            y_test,
            y_pred_rf
        ),
        4
    )
)

print(
    "Balanced Accuracy:",
    round(
        balanced_accuracy_score(
            y_test,
            y_pred_rf
        ),
        4
    )
)

print(
    "Macro F1:",
    round(
        f1_score(
            y_test,
            y_pred_rf,
            average="macro"
        ),
        4
    )
)

print(
    "\nClassification Report:\n"
)

print(
    classification_report(
        y_test,
        y_pred_rf
    )
)

Accuracy: 0.965
Balanced Accuracy: 0.8536
Macro F1: 0.9017

Classification Report:

              precision    recall  f1-score   support

     at-risk       0.97      1.00      0.98    118512
         fit       0.95      0.79      0.87      7961
   unhealthy       0.97      0.77      0.86     11545

    accuracy                           0.96    138018
   macro avg       0.96      0.85      0.90    138018
weighted avg       0.96      0.96      0.96    138018



In [66]:
boosting_models = {

    

    "XGBoost":
        XGBClassifier(
            n_estimators=300,
            learning_rate=0.1,
            max_depth=8,
            subsample=0.8,
            colsample_bytree=0.8,
            objective="multi:softprob",
            eval_metric="mlogloss",
            n_jobs=-1,
            random_state=42
        ),
    "XGBoost_Base": XGBClassifier()
}

In [68]:
import time

In [70]:
from sklearn.metrics import (
    accuracy_score,
    balanced_accuracy_score,
    precision_score,
    recall_score,
    f1_score
)

In [71]:
boosting_results = []

trained_boosting_models = {}


for model_name, model in (
    boosting_models.items()
):

    print(
        f"Training {model_name}..."
    )

    start_time = time.time()

    model.fit(
        X_train_processed,
        y_train_encoded
    )

    y_pred = model.predict(
        X_test_processed
    )

    y_pred = (
        np.asarray(y_pred)
        .reshape(-1)
        .astype(int)
    )

    training_time = (
        time.time()
        - start_time
    )

    boosting_results.append({

        "Model":
            model_name,

        "Accuracy":
            accuracy_score(
                y_test_encoded,
                y_pred
            ),

        "Balanced Accuracy":
            balanced_accuracy_score(
                y_test_encoded,
                y_pred
            ),

        "Macro Precision":
            precision_score(
                y_test_encoded,
                y_pred,
                average="macro",
                zero_division=0
            ),

        "Macro Recall":
            recall_score(
                y_test_encoded,
                y_pred,
                average="macro",
                zero_division=0
            ),

        "Macro F1":
            f1_score(
                y_test_encoded,
                y_pred,
                average="macro",
                zero_division=0
            ),

        "Training Time":
            training_time
    })

    trained_boosting_models[
        model_name
    ] = model

    print(
        f"{model_name} completed "
        f"in {training_time:.2f} seconds.\n"
    )

Training XGBoost...
XGBoost completed in 28.97 seconds.

Training XGBoost_Base...
XGBoost_Base completed in 7.08 seconds.



In [72]:
boosting_comparison = (
    pd.DataFrame(
        boosting_results
    )
    .sort_values(
        "Macro F1",
        ascending=False
    )
    .reset_index(
        drop=True
    )
)

boosting_comparison.round(4)

,Model,Accuracy,Balanced Accuracy,Macro Precision,Macro Recall,Macro F1,Training Time
0,XGBoost,0.9659,0.8601,0.9598,0.8601,0.9048,28.9664
1,XGBoost_Base,0.9658,0.8596,0.9599,0.8596,0.9045,7.0762
